# libraries

In [ ]:
using SpecialFunctions, Revise, Random, OffsetArrays, BenchmarkTools, Statistics, ProgressMeter, Profile, LinearAlgebra, JLD2

Random.seed!(1234)

include("oggetti.jl")
include("gillespie_temp.jl")

# auxiliary


In [ ]:
function checktol(density, termalization_times; tol=0.001)
    diff = norm.(density[1:end-1] .- density[2:end], Inf) # differenza massima in valore assoluto tra 
    flag = false
    for i in 1:length(diff)
        if diff[i] < tol
            println("Il sistema ha raggiunto la stazionarietà dopo $(termalization_times[i]) unità di tempo.")
            flag = true
            return termalization_times[i]
        end

    end
    if !flag
        println("Il sistema non ha raggiunto la stazionarietà entro il tempo totale simulato.")
    end
end

# param


In [ ]:
#=
L = 10
α = 0.2
β = 0.3

q_defect1 = 0.5
defect1_length = 1
from = :center
q = lattice(L, q_defect1, defect1_length, from)

tasep = Tasep(L, α, β, q)
cond_init = (0.5, 0.0);
=#

# termalization via numerical average / n_simu


In [ ]:
function singol_run(tasep, cond_init; termalization_times=collect(0:10:100), processi_nn=nn(tasep.L))
    # initialization
    t = 0.0
    mean_currents_vector = []
    mean_P10_vector = []
    configurations = [] # holds the configurations measured at the termalization times of a single run

    # system initialization
    ν, W, q = Tasep2Sys(tasep, cond_init[1])

    # PRE-ALLOCATE cumsum only once per simulation
    # (not on every iteration)
    C_undef = Vector{Float64}(undef, tasep.L + 1)
    C = OffsetArray(C_undef, 0:tasep.L)

    # Compute the initial cumsum
    cumsum!(C, W)
    for termalization_time in termalization_times
        τ = 0.0
        while t < termalization_time
            τ, event = onestep_gillespie(C)
            t = t + τ
            # update sys
            updatesys!(ν, event, tasep.L)
            # update rates
            updaterates_nn!(W, processi_nn[event+1], ν, q)
            updatecumulative!(C, W, event) # only update part of the cumulative array, from event onward
        end
        push!(mean_currents_vector, mean(copy(W ./ tasep.q)[1:end-1])) # W/q = q*ν*(1-ν) == J/q = P10
        push!(mean_P10_vector, mean(copy(W)[1:end-1])) # W = q*ν*(1-ν) == J
        push!(configurations, copy(ν[1:end-1])) # save the configuration (the densities) at the termalization time
    end

    return mean_currents_vector, mean_P10_vector, configurations
end

In [ ]:
function termalization(tasep, cond_init; termalization_times=collect(0:100:10000), n_simu=1000, tol1=0.01, tol2=0.1)
    ALL_MEAN_CURRENTS = []
    ALL_MEAN_P10 = []
    SUM_CONFIGURATIONS = [OffsetArray(zeros(tasep.L), 1:tasep.L) for i in 1:length(termalization_times)]

    @showprogress for n in 1:n_simu
        run_mean_currents, run_mean_P10, configurations = singol_run(tasep, cond_init; termalization_times=termalization_times, processi_nn=nn(tasep.L))
        push!(ALL_MEAN_CURRENTS, run_mean_currents)
        push!(ALL_MEAN_P10, run_mean_P10)
        SUM_CONFIGURATIONS = SUM_CONFIGURATIONS .+ configurations
    end


    mean_currents = mean(ALL_MEAN_CURRENTS)
    mean_P10 = mean(ALL_MEAN_P10)
    mean_configurations = SUM_CONFIGURATIONS ./ n_simu

    # compute termalization times
    J_staz = mean_currents[end]
    gap = abs.(mean_currents .- J_staz) #  ./ J_staz
    T_term4current = termalization_times[findfirst(gap .< tol1)]
    P10_staz = mean_P10[end]
    gap = abs.(mean_P10 .- P10_staz) #  ./ P10_staz
    T_term4P10 = termalization_times[findfirst(gap .< tol1)]

    T_term4densityprofile = checktol(mean_configurations, termalization_times; tol=tol2)


    return T_term4current, T_term4P10, T_term4densityprofile, mean_currents, mean_P10, mean_configurations
end

# plot

In [ ]:
function plot_termalization_summary(
    termalization_times,
    mean_currents,
    mean_P10,
    mean_configurations;
    T_term4current=nothing,
    T_term4P10=nothing,
    T_term4densityprofile=nothing,
    idx_t2=2
)
    # index of the "true" termalization time for the density profile
    idx_term = if T_term4densityprofile === nothing
        length(termalization_times)
    else
        findfirst(==(T_term4densityprofile), termalization_times)
    end
    idx_term = idx_term === nothing ? length(termalization_times) : idx_term

    # plot averaged currents
    p1 = plot(
        termalization_times,
        mean_currents,
        lw=2,
        marker=:circle,
        label="Mean current",
        xlabel="Measurement times",
        ylabel="Mean current"
    )
    if T_term4current !== nothing
        vline!(p1, [T_term4current], label="T_term current= $(T_term4current)", ls=:dash, c=:red)
    end
    if T_term4P10 !== nothing
        vline!(p1, [T_term4P10], label="T_term P10= $(T_term4P10)", ls=:dash, c=:green)
    end
    if T_term4densityprofile !== nothing
        vline!(p1, [T_term4densityprofile], label="T_term density= $(T_term4densityprofile)", ls=:dash, c=:blue)
    end

    # plot mean P10
    p2 = plot(
        termalization_times,
        mean_P10,
        lw=2,
        marker=:circle,
        label="Mean P10",
        xlabel="Measurement times",
        ylabel="Mean P10"
    )
      if T_term4current !== nothing
        vline!(p2, [T_term4current], label="T_term current= $(T_term4current)", ls=:dash, c=:red)
    end
    if T_term4P10 !== nothing
        vline!(p2, [T_term4P10], label="T_term P10= $(T_term4P10)", ls=:dash, c=:green)
    end
    if T_term4densityprofile !== nothing
        vline!(p2, [T_term4densityprofile], label="T_term density= $(T_term4densityprofile)", ls=:dash, c=:blue)
    end

    # density profiles
    prof2 = collect(mean_configurations[idx_t2])
    profT = collect(mean_configurations[idx_term])
    profL = collect(mean_configurations[end])

    x2 = 1:length(prof2)
    xT = 1:length(profT)
    xL = 1:length(profL)

    p3 = plot(
        x2, prof2,
        lw=2,
        label="density al tempo iniziale ($(termalization_times[idx_t2]))",
        xlabel="Site",
        ylabel="Density",
    )

    p4 = plot(
        xT, profT,
        lw=2,
        label="density al T_term ($(termalization_times[idx_term]))",
        xlabel="Site",
    )

    p5 = plot(
        xL, profL,
        lw=2,
        label=" density at the last time ($(termalization_times[end]))",
        xlabel="Site",
    )
    
    pd = plot(p3, p4, p5,
        layout=(1, 3))

    return plot(p1, p2, pd, layout=(3, 1), size=(1000, 700))
end

### low density phase

In [ ]:
L = 200
α = 0.2
β = 0.3

q_defect1 = 0.5
defect1_length = 1
from = :center
q = lattice(L, q_defect1, defect1_length, from)

tasep = Tasep(L, α, β, q)
cond_init = (0.0, 0.0);

termalization_times = vcat(collect(0:100:1000), collect(1000:200:2_000),collect(2000:500:10_000))
n_simu = 1e5
tol_current_P10, tol_densityprofile = 0.001 , 0.001
termalization_time  = collect(0:100:10_000)
tol_densityprofile = 0.0001

#T_term4current, T_term4P10, T_term4densityprofile, mean_currents, mean_P10, mean_configurations = termalization(tasep, cond_init; termalization_times=termalization_times, n_simu=n_simu, tol1=tol_current_P10, tol2=tol_densityprofile)
#@save "data/termalization/term_L200_lowph_nsimu_1e5.jld2" T_term4current T_term4P10 T_term4densityprofile mean_currents mean_P10 mean_configurations
#@load "data/termalization/term_L100_lowph_nsimu_1e5.jld2" T_term4current T_term4P10 T_term4densityprofile mean_currents mean_P10 mean_configurations

plot_termalization_summary(
    termalization_times,
    mean_currents,
    mean_P10,
    mean_configurations;
    T_term4current=T_term4current,
    T_term4P10=T_term4P10,
    T_term4densityprofile=T_term4densityprofile,
    idx_t2=2
)

In [ ]:
    T_term4densityprofile = checktol(mean_configurations, termalization_times; tol=0.001)


In [ ]:
plot(mean_configurations[findfirst(termalization_times .== 700)], size=(800, 250), label="density profile at the termalization time", xlabel="Site", ylabel="Density")

## high phase

In [ ]:
L = 200
α = 0.3
β = 0.2

q_defect1 = 0.5
defect1_length = 1
from = :center
q = lattice(L, q_defect1, defect1_length, from)

tasep = Tasep(L, α, β, q)
cond_init = (0.0, 0.0);

termalization_times = vcat(collect(0:100:1000), collect(1000:200:2_000),collect(2000:500:10_000))
n_simu = 1e5
tol_current_P10, tol_densityprofile = 0.001 , 0.001
termalization_time  = collect(0:100:10_000)


#T_term4current, T_term4P10, T_term4densityprofile, mean_currents, mean_P10, mean_configurations = termalization(tasep, cond_init; termalization_times=termalization_times, n_simu=n_simu, tol1=tol_current_P10, tol2=tol_densityprofile)
#@save "data/termalization/term_L200_hiph_nsimu_1e5.jld2" T_term4current T_term4P10 T_term4densityprofile mean_currents mean_P10 mean_configurations
#@load "data/termalization/term_L100_lowph_nsimu_1e5.jld2" T_term4current T_term4P10 T_term4densityprofile mean_currents mean_P10 mean_configurations

plot_termalization_summary(
    termalization_times,
    mean_currents,
    mean_P10,
    mean_configurations;
    T_term4current=T_term4current,
    T_term4P10=T_term4P10,
    T_term4densityprofile=T_term4densityprofile,
    idx_t2=2
)

## coe ph

In [ ]:
L = 200
α = 0.3
β = 0.3

q_defect1 = 0.5
defect1_length = 1
from = :center
q = lattice(L, q_defect1, defect1_length, from)

tasep = Tasep(L, α, β, q)
cond_init = (0.0, 0.0);

termalization_times = vcat(collect(0:100:1000),collect(1500:500:7_000))
n_simu = 1e5
tol_current_P10, tol_densityprofile = 0.001 , 0.001


T_term4current, T_term4P10, T_term4densityprofile, mean_currents, mean_P10, mean_configurations = termalization(tasep, cond_init; termalization_times=termalization_times, n_simu=n_simu, tol1=tol_current_P10, tol2=tol_densityprofile)
@save "data/termalization/term_L200_coeph_nsimu_1e5.jld2" T_term4current T_term4P10 T_term4densityprofile mean_currents mean_P10 mean_configurations
#@load "data/termalization/term_L200_coeph_nsimu_1e5.jld2" T_term4current T_term4P10 T_term4densityprofile mean_currents mean_P10 mean_configurations

plot_termalization_summary(
    termalization_times,
    mean_currents,
    mean_P10,
    mean_configurations;
    T_term4current=T_term4current,
    T_term4P10=T_term4P10,
    T_term4densityprofile=T_term4densityprofile,
    idx_t2=2
)

## mc ph, defect length 1, q = 1


In [ ]:
L = 100
α = 0.6
β = 0.7

q_defect1 = 1
defect1_length = 1
from = :center
q = lattice(L, q_defect1, defect1_length, from)

tasep = Tasep(L, α, β, q)
cond_init = (0.0, 0.0);

termalization_times = vcat(collect(0:100:1000), collect(1000:200:2_000),collect(2000:1000:10_000))

n_simu = 1e5
tol_current_P10, tol_densityprofile = 0.001 , 0.001


#T_term4current, T_term4P10, T_term4densityprofile, mean_currents, mean_P10, mean_configurations = termalization(tasep, cond_init; termalization_times=termalization_times, n_simu=n_simu, tol1=tol_current_P10, tol2=tol_densityprofile)
#@save "data/termalization/term_L200_mcph_nsimu_1e5_Ldef_1_q=1.jld2" T_term4current T_term4P10 T_term4densityprofile mean_currents mean_P10 mean_configurations
@load "data/termalization/term_L200_mcph_nsimu_1e5_Ldef_1_q=1.jld2" T_term4current T_term4P10 T_term4densityprofile mean_currents mean_P10 mean_configurations



In [ ]:
plot_termalization_summary(
    termalization_times,
    mean_currents,
    mean_P10,
    mean_configurations;
    T_term4current=T_term4current,
    T_term4P10=T_term4P10,
    T_term4densityprofile=T_term4densityprofile,
    idx_t2=2
)

In [ ]:
plot(mean_P10)

In [ ]:
termalization_times = vcat(collect(0:100:1000),collect(1500:500:7_000))

T1 = 2500
T2 = 6999
plot(mean_configurations[findfirst(termalization_times .> T1)], label= "$T1", xlabel="Site", ylabel="Density")
plot!(mean_configurations[findfirst(termalization_times .> T2)], label= "$T2", xlabel="Site", ylabel="Density")

## mc ph, defect length 1, q = 0.1


In [ ]:
L = 100
α = 0.6
β = 0.7

q_defect1 = 0.1
defect1_length = 1
from = :center
q = lattice(L, q_defect1, defect1_length, from)

tasep = Tasep(L, α, β, q)
cond_init = (0.0, 0.0);

termalization_times = vcat(collect(0:100:1000),collect(1500:500:7_000))
n_simu = 1e5
tol_current_P10, tol_densityprofile = 0.001 , 0.001



#T_term4current, T_term4P10, T_term4densityprofile, mean_currents, mean_P10, mean_configurations = termalization(tasep, cond_init; termalization_times=termalization_times, n_simu=n_simu, tol1=tol_current_P10, tol2=tol_densityprofile)
#@save "data/termalization/term_L100_mcph_nsimu_1e5_Ldef_1_q=0.1.jld2" T_term4current T_term4P10 T_term4densityprofile mean_currents mean_P10 mean_configurations
@load "data/termalization/term_L100_mcph_nsimu_1e5_Ldef_1_q=0.1.jld2" T_term4current T_term4P10 T_term4densityprofile mean_currents mean_P10 mean_configurations

plot_termalization_summary(
    termalization_times,
    mean_currents,
    mean_P10,
    mean_configurations;
    T_term4current=T_term4current,
    T_term4P10=T_term4P10,
    T_term4densityprofile=T_term4densityprofile,
    idx_t2=2
)

## mc ph, defect length 8, q = 0.1


In [ ]:
L = 200
α = 0.6
β = 0.7

q_defect1 = 0.1
defect1_length = 8
from = :center
q = lattice(L, q_defect1, defect1_length, from)

tasep = Tasep(L, α, β, q)
cond_init = (0.0, 0.0);

termalization_times = vcat(collect(0:100:1000),collect(1500:500:7_000))
n_simu = 1e5
tol_current_P10, tol_densityprofile = 0.001 , 0.001


#T_term4current, T_term4P10, T_term4densityprofile, mean_currents, mean_P10, mean_configurations = termalization(tasep, cond_init; termalization_times=termalization_times, n_simu=n_simu, tol1=tol_current_P10, tol2=tol_densityprofile)
#@save "data/termalization/term_L200_mcph_nsimu_1e5_Ldef_8_q=0.1.jld2" T_term4current T_term4P10 T_term4densityprofile mean_currents mean_P10 mean_configurations
@load "data/termalization/term_L200_mcph_nsimu_1e5_Ldef_8_q=0.1.jld2" T_term4current T_term4P10 T_term4densityprofile mean_currents mean_P10 mean_configurations

plot_termalization_summary(
    termalization_times,
    mean_currents,
    mean_P10,
    mean_configurations;
    T_term4current=T_term4current,
    T_term4P10=T_term4P10,
    T_term4densityprofile=T_term4densityprofile,
    idx_t2=2
)

# end